In [9]:
import numpy as np
import gseapy as gp
from collections import defaultdict

In [10]:
gene_pairs_f_final = np.load('gene_pairs_f_final.npy')
gene_pairs_r_final= np.load('gene_pairs_r_final.npy')

In [11]:
gene_sets=['ChEA_2022', 'ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X',\
           'TF_Perturbations_Followed_by_Expression','Reactome_2022', 'BioPlanet_2019',\
           'WikiPathway_2023_Human', ]
pairs_terms_f, pair_annotations_f= {}, {}
for gene_pair in gene_pairs_f_final:
    gene_list = gene_pair.split()
    print(gene_list)
    enr = gp.enrichr(gene_list=gene_list, # or "./tests/data/gene_list.txt",
                     gene_sets=gene_sets,
                     #organism='human', # don't forget to set organism to the one you desired! e.g. Yeast
                     outdir=None, # don't write to disk
                    )
#     print(enr.results['Genes'])

    term_enr = enr.results[enr.results['Genes']== gene_list[0]+';'+gene_list[1]][enr.results['Adjusted P-value']<=.05]['Term']
    
   
    pairs_terms_f[gene_pair] = term_enr
    
    if len(term_enr)== 0:
        print('none found')
    else:
        terms_single = []
        for i in range(len(term_enr)):
            terms_single.append(term_enr.iloc[i].split()[0])
        pair_annotations_f[gene_pair] = set(terms_single)
        print(term_enr)
        print(pair_annotations_f[gene_pair])
        

['EDC4', 'NSD3']


Exception: Error sending gene list, try again later

In [12]:
pairs_terms_r, pair_annotations_r= {}, {}
for gene_pair in gene_pairs_r_final:
    gene_list = gene_pair.split()
    print(gene_list)
    enr = gp.enrichr(gene_list=gene_list, # or "./tests/data/gene_list.txt",
                     gene_sets=gene_sets,
                     organism='human', # don't forget to set organism to the one you desired! e.g. Yeast
                     outdir=None, # don't write to disk
                    )

    term_enr = enr.results[enr.results['Genes']== gene_list[0]+';'+gene_list[1]][enr.results['Adjusted P-value']<=.05]['Term']
    
   
    pairs_terms_r[gene_pair] = term_enr
    
    if len(term_enr)== 0:
        print('none found')
    else:
        terms_single = []
        for i in range(len(term_enr)):
            terms_single.append(term_enr.iloc[i].split()[0])
        pair_annotations_r[gene_pair] = set(terms_single)
        print(term_enr)
        print(pair_annotations_r[gene_pair])

['JAK2', 'DPPA5']


Exception: Error sending gene list, try again later

In [13]:
# Task 1: Find common genes and gene pairs between pair_annotations_f and pair_annotations_r
common_genes = set(pair_annotations_f.keys()) & set(pair_annotations_r.keys())
print("Common genes and gene pairs:", common_genes)

# Break down gene pairs into individual genes for pair_annotations_f
list_f = []
for gene_pair in pair_annotations_f.keys():
    genes = gene_pair.split()
    list_f.extend(genes)

# Break down gene pairs into individual genes for pair_annotations_r
list_r = []
for gene_pair in pair_annotations_r.keys():
    genes = gene_pair.split()
    list_r.extend(genes)

# Find common genes between list_f and list_r
common_genes = set(list_f) & set(list_r)
print("Common genes:", common_genes)




# Create lists of all descriptors for every gene pair from both dictionaries
all_descriptors_f = [descriptors for descriptors in pair_annotations_f.values()]
all_descriptors_r = [descriptors for descriptors in pair_annotations_r.values()]

# Flatten the lists of descriptors
flat_descriptors_f = [item for sublist in all_descriptors_f for item in sublist]
flat_descriptors_r = [item for sublist in all_descriptors_r for item in sublist]
flat_descriptors_f, flat_descriptors_r = set(flat_descriptors_f), set(flat_descriptors_r)
# Find common descriptors between both dictionaries
common_descriptors = set(flat_descriptors_f) & set(flat_descriptors_r)

# Initialize defaultdict to store gene pairs for each common descriptor
common_descriptors_dict, common_descriptors_dict_f, common_descriptors_dict_r = defaultdict(list), defaultdict(list), defaultdict(list)

print('Common descriptors between forward and reverse protocols')
# Iterate over each common descriptor
for descriptor in common_descriptors:
    # Iterate over gene pairs in pair_annotations_f
    for gene_pair, descriptors in pair_annotations_f.items():
        if descriptor in descriptors:
            common_descriptors_dict[descriptor].append(gene_pair)

    # Iterate over gene pairs in pair_annotations_r
    for gene_pair, descriptors in pair_annotations_r.items():
        if descriptor in descriptors:
            common_descriptors_dict[descriptor].append(gene_pair)


    
# print('Common descriptors in forward protocols')
# # Iterate over each common descriptor
for descriptor in flat_descriptors_f:
    # Iterate over gene pairs in pair_annotations_f
    for gene_pair, descriptors in pair_annotations_f.items():
        if descriptor in descriptors:
            common_descriptors_dict_f[descriptor].append(gene_pair)
# Iterate over each common descriptor
for descriptor in flat_descriptors_r:
    # Iterate over gene pairs in pair_annotations_f
    for gene_pair, descriptors in pair_annotations_r.items():
        if descriptor in descriptors:
            common_descriptors_dict_r[descriptor].append(gene_pair)


    
    
# Task 3: Make a ranked dictionary of descriptors based on occurrence and gene pairs
descriptor_rank = defaultdict(lambda: {'count': 0, 'pairs': []})
for descriptors, gene_pairs in common_descriptors_dict.items():
    descriptor_rank[descriptors]['count'] = len(gene_pairs)
    descriptor_rank[descriptors]['pairs'] = gene_pairs

# Sort the descriptors by occurrence count in descending order
sorted_descriptors = sorted(descriptor_rank.items(), key=lambda x: x[1]['count'], reverse=True)
ranked_descriptor_dict = {descriptor: info['count'] for descriptor, info in sorted_descriptors}

# Print the ranked dictionary of descriptors
print("\nRanked dictionary of descriptors:")
for descriptor, count in ranked_descriptor_dict.items():
    print(f"{descriptor}: {count} times")

# Additional: Print gene pairs associated with each descriptor
print("\nGene pairs associated with each descriptor:")
for descriptor, info in sorted_descriptors:
    print(f"{descriptor}: {info['pairs']}")
    
    
descriptor_rank = defaultdict(lambda: {'count': 0, 'pairs': []})
for descriptors, gene_pairs in common_descriptors_dict_f.items():
    descriptor_rank[descriptors]['count'] = len(gene_pairs)
    descriptor_rank[descriptors]['pairs'] = gene_pairs

# Sort the descriptors by occurrence count in descending order
sorted_descriptors = sorted(descriptor_rank.items(), key=lambda x: x[1]['count'], reverse=True)
ranked_descriptor_dict = {descriptor: info['count'] for descriptor, info in sorted_descriptors}

# Print the ranked dictionary of descriptors
print("\nRanked dictionary of descriptors Forward:")
for descriptor, count in ranked_descriptor_dict.items():
    print(f"{descriptor}: {count} times")

# Additional: Print gene pairs associated with each descriptor
print("\nGene pairs associated with each descriptor Forward:")
for descriptor, info in sorted_descriptors:
    print(f"{descriptor}: {info['pairs']}")
    
    
descriptor_rank = defaultdict(lambda: {'count': 0, 'pairs': []})
for descriptors, gene_pairs in common_descriptors_dict_r.items():
    descriptor_rank[descriptors]['count'] = len(gene_pairs)
    descriptor_rank[descriptors]['pairs'] = gene_pairs

# Sort the descriptors by occurrence count in descending order
sorted_descriptors = sorted(descriptor_rank.items(), key=lambda x: x[1]['count'], reverse=True)
ranked_descriptor_dict = {descriptor: info['count'] for descriptor, info in sorted_descriptors}

# Print the ranked dictionary of descriptors
print("\nRanked dictionary of descriptors Reverse:")
for descriptor, count in ranked_descriptor_dict.items():
    print(f"{descriptor}: {count} times")

# Additional: Print gene pairs associated with each descriptor
print("\nGene pairs associated with each descriptor Reverse:")
for descriptor, info in sorted_descriptors:
    print(f"{descriptor}: {info['pairs']}")
    
    




Common genes and gene pairs: set()
Common genes: set()
Common descriptors between forward and reverse protocols

Ranked dictionary of descriptors:

Gene pairs associated with each descriptor:

Ranked dictionary of descriptors Forward:

Gene pairs associated with each descriptor Forward:

Ranked dictionary of descriptors Reverse:

Gene pairs associated with each descriptor Reverse:
